## Complete Dictionary with included events as desired - May be exported for one time completion 

In [1]:
import os
import re
import pathlib as pl

In [2]:
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy23'

In [3]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'event_tie_ins'

home is at:  c:\_code\hms_to_ras_sst


In [4]:
#create folder to export the tie-in dictionaries.
if not os.path.exists(export_folder):
    os.makedirs(export_folder)

In [5]:
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)

In [7]:
#list of all applicable HUC10s
huc_connect_huc.keys()

complete_dictionary_p1 = {}

for huc in huc_connect_huc.keys():
    prob_shps = glob.glob(str(inputs/project/'hms_ras'/huc/'*all_junctions_events.geojson'))
    complete_dictionary_p1.update(get_sst_storms_by_recurrence(huc,prob_shps))

IndexError: list index out of range

In [7]:
#manual adjustments required for some of the HUC models (ds_junc does not fall within the plot and thus it cannot find the events related to it. 
#The dictionary requires that the upstream junction be used. select one in similar stream as the ds_junc)

ds_junc_adjustments = {'1404010109':'HUC_101_J_277',
                       '1404010401':'HUC_104_J_307',
                       '1404010602':'HUC_106_J_42',
                       '1404010903':'HUC_109_J_156',
                       '1404010705':'HUC_107_J_220'} 



In [10]:
#create separate temporary dictionary to add to the list shown in the events
completed_us_dictionary_p2 = {}

for huc in huc_connect_huc.keys():
    us_huc_temp = [k for k,v in huc_connect_huc.items() if v == huc]
    if len(us_huc_temp) == 0:
        print(f'huc {huc} is most upstream')
    else:
        bc_connections = {'junctions':{},'dss_path':{},'ts':{}}
        
        us_hucs = [k for k in us_huc_temp]
        ds_j = [huc_connect_j[k] for k in us_huc_temp]
        
        for us in us_hucs:
            if us not in ds_junc_adjustments.keys():
                bc_connections['junctions'][us] = huc_connect_j[us]
                bc_connections['dss_path'][us] = {}
            else:
                bc_connections['junctions'][us] = ds_junc_adjustments[us]
                bc_connections['dss_path'][us] = {}
                
        events_dict_us = {}
        print(f'filling out dictionary for huc {huc}')
        for huc, j in  bc_connections['junctions'].items():
            events_dict_us[huc] = {j:{}}
            htmls = glob.glob(str(inputs/project/huc[4:8]/f'HUC{huc[:8]}'/huc/'plots'/'*_map.html'))
            events_dict_us = get_sst_storms_by_recurrence_us_huc(events_dict_us,huc,htmls,j)
            
        completed_us_dictionary_p2.update(events_dict_us)

len(completed_us_dictionary_p2)

huc 1404010102 is most upstream
huc 1404010103 is most upstream
huc 1404010104 is most upstream
huc 1404010105 is most upstream
huc 1404010106 is most upstream
filling out dictionary for huc 1404010107
huc 1404010108 is most upstream
filling out dictionary for huc 1404010109
huc 1404010110 is most upstream
filling out dictionary for huc 1404010111
huc 1404010112 is most upstream
filling out dictionary for huc 1404010113
huc 1404010201 is most upstream
huc 1404010202 is most upstream
huc 1404010203 is most upstream
huc 1404010204 is most upstream
huc 1404010205 is most upstream
filling out dictionary for huc 1404010206
huc 1404010301 is most upstream
huc 1404010302 is most upstream
filling out dictionary for huc 1404010303
huc 1404010304 is most upstream
huc 1404010305 is most upstream
filling out dictionary for huc 1404010306
huc 1404010401 is most upstream
huc 1404010402 is most upstream
filling out dictionary for huc 1404010403
huc 1404010404 is most upstream
filling out dictionary f

59

In [ ]:
import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'completed_event_dictionary.json', "w") as outfile:
    json.dump(complete_dictionary_p1, outfile, indent= 1)

In [ ]:
#WARNING --- This dictionary has had the downstream junctions modified for a select number of HUCs in order to attain the events related to it. Be advised they will not match ds junctions 100%

import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'tie_in_dictionary.json', "w") as outfile2:
    json.dump(completed_us_dictionary_p2, outfile2, indent= 1)